# 01. Data Cleaning & Preprocessing (Walmart Sales Analysis)

📌 **Executive Summary**
Before conducting Exploratory Data Analysis (EDA) and predictive modeling, this notebook ensures data integrity and consistency by addressing missing values, data anomalies, and multi-table integration.

**Key Preprocessing Steps:**
1. **Anomaly Audit**: Identified negative sales values attributable to customer returns and refunds.
2. **Missing Value Imputation**:
   - Replaced `NaN` values in `MarkDown1~5` with `0` (indicating no promotional discounts applied).
   - Applied forward-fill interpolation (`ffill`) on `CPI` and `Unemployment` indicators to maintain time-series continuity.
3. **Feature Engineering**: Extracted temporal features (`Year`, `Month`, `Week`) from `Date` for granular trend analysis.
4. **Data Integration**: Left-joined `train.csv`, `stores.csv`, and `features.csv` while auditing for zero row-loss integrity.

### Step 1: Environment Setup & Raw Data Loading
Mounting Google Drive and loading raw datasets (`train.csv`, `stores.csv`, `features.csv`).

### Step 2: Anomaly & Sales Data Audit
Auditing `Weekly_Sales` for negative entries (returns/refunds) and zero-sale records to understand transaction boundaries.

### Step 3: Missing Value Treatment & Imputation
Handling structural missing values across promotional markdowns and economic indices (`CPI`, `Unemployment`).

### Step 4: Datetime Conversion, Feature Engineering & Table Joining
Converting `Date` into standard datetime format, engineering calendar features, and joining tables on `Store` and `Date` keys with row-loss verification.

In [ ]:
# ==========================================
# 1. Mount Google Drive & Import Libraries
# ==========================================
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/포트폴리오-walmart(캐글)/'

# Load raw datasets
train_df = pd.read_csv(DATA_PATH + 'train.csv')
stores_df = pd.read_csv(DATA_PATH + 'stores.csv')
features_df = pd.read_csv(DATA_PATH + 'features.csv')

print("==========================================")
print(" 📊 DATA CLEANING & AUDIT REPORT")
print("==========================================")

# ------------------------------------------
# Audit 1: Check Negative & Zero Sales
# ------------------------------------------
neg_sales_cnt = (train_df['Weekly_Sales'] < 0).sum()
zero_sales_cnt = (train_df['Weekly_Sales'] == 0).sum()

print(f"[1] Weekly_Sales Anomaly Check")
print(f"    - Negative Sales (Returns/Refunds) : {neg_sales_cnt:,} records")
print(f"    - Zero Sales Records               : {zero_sales_cnt:,} records")

# ------------------------------------------
# Audit 2: Missing Value Treatment
# ------------------------------------------
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
md_null_before = features_df[markdown_cols].isnull().sum().sum()

# Fill missing MarkDown values with 0 (No Promotion)
features_df[markdown_cols] = features_df[markdown_cols].fillna(0)

cpi_null_before = features_df['CPI'].isnull().sum()
unemp_null_before = features_df['Unemployment'].isnull().sum()

# Interpolate CPI and Unemployment using Forward Fill
features_df['CPI'] = features_df['CPI'].interpolate(method='ffill')
features_df['Unemployment'] = features_df['Unemployment'].interpolate(method='ffill')

print(f"\n[2] Missing Value Imputation Summary")
print(f"    - MarkDown1~5 Missing Values Replaced (with 0) : {md_null_before:,} records")
print(f"    - CPI Missing Values Interpolated              : {cpi_null_before:,} records")
print(f"    - Unemployment Missing Values Interpolated     : {unemp_null_before:,} records")

# ------------------------------------------
# Audit 3: Datetime Conversion & Feature Engineering
# ------------------------------------------
train_df['Date'] = pd.to_datetime(train_df['Date'])
features_df['Date'] = pd.to_datetime(features_df['Date'])

train_df['Year'] = train_df['Date'].dt.year
train_df['Month'] = train_df['Date'].dt.month
train_df['Week'] = train_df['Date'].dt.isocalendar().week

# ------------------------------------------
# Audit 4: Data Merge & Integrity Validation
# ------------------------------------------
merged_df = pd.merge(train_df, stores_df, on='Store', how='left')
final_df = pd.merge(merged_df, features_df, on=['Store', 'Date', 'IsHoliday'], how='left')

row_diff = len(train_df) - len(final_df)
print(f"\n[3] Data Merge Integrity Audit")
print(f"    - Raw train_df Row Count   : {len(train_df):,}")
print(f"    - Merged final_df Row Count: {len(final_df):,}")
print(f"    - Row Count Discrepancy    : {row_diff} (0 indicates a perfect join)")

# ------------------------------------------
# Save Cleaned Dataset
# ------------------------------------------
SAVE_PATH = DATA_PATH + 'clean_walmart_data.csv'
final_df.to_csv(SAVE_PATH, index=False)

print(f"\n[4] Cleaned Data Saved Successfully: 'clean_walmart_data.csv'")
print("==========================================")